# Práctica 4: Ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

### Ejercicio 1

La biblioteca `Unified Planning` permite leer un dominio de planificación automática a partir de un fichero PDDL y crear posteriormente instancias de ese dominio mediante la interfaz proporcionada por la biblioteca.

En este ejercicio se pretende seguir esa metodología para crear instancias del mundo de los bloques que contengan los bloques $B_{0}$ a $B_{N - 1}$, apilados inicialmente en ese orden, y en la que el objetivo sea que estén apilados al contrario, con el bloque $B_{N - 1}$ sobre la mesa, el bloque $B_{N - 2}$ sobre el $B_{N - 1}$, el $B_{N - 3}$ sobre el $B_{N - 2}$, etc.

Se pide realizar lo siguiente:

1. Leer el dominio del mundo de los bloques a partir del fichero `dominio_mundo_bloques.pddl`.

In [1]:
from unified_planning.io import PDDLReader

In [2]:
lector_PDDL = PDDLReader()
dominio_mundo_bloques = lector_PDDL.parse_problem("dominio_mundo_bloques.pddl", None)

In [ ]:
print(dominio_mundo_bloques) 


problem name = dominio_mundo_bloques

types = [object]

fluents = [
  bool sobre_la_mesa[b=object]
  bool sobre[b1=object, b2=object]
  bool agarrado[b=object]
  bool brazo_libre
  bool despejado[b=object]
]

actions = [
  action agarrar(object b) {
    preconditions = [
      (sobre_la_mesa(b) and despejado(b) and brazo_libre)
    ]
    effects = [
      sobre_la_mesa(b) := false
      despejado(b) := false
      brazo_libre := false
      agarrado(b) := true
    ]
  }
  action bajar(object b) {
    preconditions = [
      agarrado(b)
    ]
    effects = [
      agarrado(b) := false
      sobre_la_mesa(b) := true
      despejado(b) := true
      brazo_libre := true
    ]
  }
  action desapilar(object b1, object b2) {
    preconditions = [
      (sobre(b1, b2) and despejado(b1) and brazo_libre)
    ]
    effects = [
      sobre(b1, b2) := false
      despejado(b1) := false
      brazo_libre := false
      agarrado(b1) := true
      despejado(b2) := true
    ]
  }
  action apilar(object

2. Completar la definición de la función `crea_instancia_mundo_bloques`, sustituyendo los `...` por código adecuado para que, dado el número `N` de bloques, la función proporcione el problema del mundo de los bloques descrito anteriormente.

In [7]:
from unified_planning.shortcuts import *

In [41]:
def crea_instancia_mundo_bloques(N):
    instancia = dominio_mundo_bloques.clone()  # Trabajamos con una copia del dominio
    # Se añaden los objetos de la instancia
    block_type = instancia.user_type('object')
    
    # Solución: Creamos los objetos 
    instancia.add_objects([ Object(f"B{i}",block_type) for i in range(N)  ])
    
    # Se establece el estado inicial de la instancia
    sobre_la_mesa = dominio_mundo_bloques.fluent('sobre_la_mesa')
    sobre = dominio_mundo_bloques.fluent('sobre')
    agarrado = dominio_mundo_bloques.fluent('agarrado')
    brazo_libre = dominio_mundo_bloques.fluent('brazo_libre')
    despejado = dominio_mundo_bloques.fluent('despejado')
    
    lista_bloques = list(instancia.all_objects)
    # Brazo libre al inicio
    instancia.set_initial_value(brazo_libre(), True)

    # B0 sobre la mesa, B1 sobre B0, B2 sobre B1,... BN sobre BN-1
    bloque_anterior = None
    for bloque in lista_bloques:
        if bloque_anterior == None:
            instancia.set_initial_value(sobre_la_mesa(bloque), True)
        else:
            instancia.set_initial_value(sobre(bloque, bloque_anterior), True)
        
        bloque_anterior = bloque

    # último bloque (BN) despejado
    instancia.set_initial_value(despejado(lista_bloques[-1]), True)

    # Se establece el objetivo de la instancia
    #...
    # BN sobre la mesa, BN-1 sobre BN-2,... , B1 sobre B0
    bloque_anterior = None
    for bloque in reversed(lista_bloques):
        if bloque_anterior == None:
            instancia.add_goal(sobre_la_mesa(bloque))
        else:
            instancia.add_goal(sobre(bloque, bloque_anterior))
        
        bloque_anterior = bloque

    return instancia

instancia = crea_instancia_mundo_bloques(7)


print("--- ESTADO INICIAL ---")
for fluido, valor in instancia.initial_values.items():
    # En unified_planning, los valores a veces son objetos FNode, 
    # así que podemos verificar si equivalen a True
    if str(valor) == "true" or valor is True:
        print(f"✅ {fluido}")
    elif str(valor) != "false" and valor is not False:
        # Por si tienes fluentes numéricos (ej. altura, batería)
        print(f"🔹 {fluido} = {valor}")
print("----------------------")

print("--- ESTADO OBJETIVO ---")
for meta in instancia.goals:
    print(f"🎯 {meta}")
print("-----------------------")


--- ESTADO INICIAL ---
✅ brazo_libre
✅ sobre_la_mesa(B0)
✅ sobre(B1, B0)
✅ sobre(B2, B1)
✅ sobre(B3, B2)
✅ sobre(B4, B3)
✅ sobre(B5, B4)
✅ sobre(B6, B5)
✅ despejado(B6)
----------------------
--- ESTADO OBJETIVO ---
🎯 sobre_la_mesa(B6)
🎯 sobre(B5, B6)
🎯 sobre(B4, B5)
🎯 sobre(B3, B4)
🎯 sobre(B2, B3)
🎯 sobre(B1, B2)
🎯 sobre(B0, B1)
-----------------------


3. Usar el planificador `Fast Downward` para tratar de resolver la instancia con el mayor número posible de bloques.

In [43]:
import time
from unified_planning.shortcuts import OneshotPlanner
from unified_planning.engines import PlanGenerationResultStatus

up.shortcuts.get_environment().credits_stream = None

# Tamaños de torre que vamos a probar
tamanos_a_probar = [5, 10, 20, 30, 50, 100, 200, 500]

print("🚀 Iniciando Prueba de Estrés - Mundo de Bloques")

for N in tamanos_a_probar:
    print(f"\n--- Probando con N = {N} bloques ---")
    instancia = crea_instancia_mundo_bloques(N)
    
    # Abrimos el planificador. 
    with OneshotPlanner(name="fast-downward") as planner:
        inicio = time.time()
        
        # Le damos un máximo de 30 segundos por problema
        resultado = planner.solve(instancia, timeout=30) 
        
        fin = time.time()
        tiempo_total = fin - inicio
        
        estado = resultado.status
        
        # Verificamos si logró encontrar una solución válida
        if estado in [PlanGenerationResultStatus.SOLVED_SATISFICING, 
                      PlanGenerationResultStatus.SOLVED_OPTIMALLY]:
            longitud_plan = len(resultado.plan.actions)
            print(f"✅ ¡Resuelto! Tiempo: {tiempo_total:.2f} segundos.")
            print(f"🔹 Acciones necesarias: {longitud_plan}")
        else:
            print(f"❌ Falló o excedió el tiempo límite (Estado: {estado.name}).")
            print("🛑 Límite de la máquina alcanzado.")
            break # Rompemos el bucle, ya no intentamos con N más grandes

🚀 Iniciando Prueba de Estrés - Mundo de Bloques

--- Probando con N = 5 bloques ---
✅ ¡Resuelto! Tiempo: 0.75 segundos.
🔹 Acciones necesarias: 10

--- Probando con N = 10 bloques ---
✅ ¡Resuelto! Tiempo: 1.41 segundos.
🔹 Acciones necesarias: 20

--- Probando con N = 20 bloques ---
✅ ¡Resuelto! Tiempo: 1.23 segundos.
🔹 Acciones necesarias: 40

--- Probando con N = 30 bloques ---
✅ ¡Resuelto! Tiempo: 2.13 segundos.
🔹 Acciones necesarias: 60

--- Probando con N = 50 bloques ---
✅ ¡Resuelto! Tiempo: 5.72 segundos.
🔹 Acciones necesarias: 100

--- Probando con N = 100 bloques ---
✅ ¡Resuelto! Tiempo: 29.72 segundos.
🔹 Acciones necesarias: 200

--- Probando con N = 200 bloques ---
❌ Falló o excedió el tiempo límite (Estado: TIMEOUT).
🛑 Límite de la máquina alcanzado.


### Ejercicio 2

En el marco de la _Conferencia Internacional sobre Planificación Automática y Planificación Temporal_ ([International Conference on Automated Planning and
Scheduling, ICAPS](http://www.icaps-conference.org/)) se celebra, con periodicidad aproximadamente trienal, la _Competición Internacional de Planificación_ (https://www.icaps-conference.org/competitions/).

Esta competición tiene diferentes objetivos: realizar una comparación empírica del estado del arte de los sistemas de planificación; destacar desafíos para la comunidad de Planificación Automática; proponer nuevas direcciones para la investigación y nuevos vínculos con otros campos de la Inteligencia Artificial; y proporcionar nuevos conjuntos de datos que puedan ser utilizados por la comunidad científica como puntos de referencia.

Uno de los dominios utilizados en la competición del año 2002 combinaba el mundo de los bloques con la distribución logística de cajas. En este dominio hay una serie de camiones (que asumimos con capacidad infinita) que transportan cajas entre distintos lugares (que asumimos que están todos conectados entre sí); en esos lugares hay unos palés, sobre los que las cajas se colocan apiladas; los apilamientos se realizan con [polipastos](https://es.wikipedia.org/wiki/Polipasto) (hay al menos uno en cada lugar).

En este ejercicio se pide completar la especificación del dominio que se proporciona a continuación, sustituyendo en las acciones los `...` por hechos adecuados, y tratar de resolver la mayor cantidad posible de las instancias de problemas proporcionadas en la carpeta Depot.

In [ ]:
from unified_planning.shortcuts import *


In [13]:
dominio_depot = Problem('Depot')



In [14]:
# Jerarquía de tipos de objetos

Place = UserType('Place')  # Lugar
Locatable = UserType('Locatable')  # Ubicable
Depot = UserType('Depot', Place)  # Almacén (Tipo de lugar)
Distributor = UserType('Distributor', Place)  # Distribuidor (Tipo de lugar)
Truck = UserType('Truck', Locatable)  # Camión (Tipo de ubicable)
Hoist = UserType('Hoist', Locatable)  # Polipasto (Tipo de ubicable)
Surface = UserType('Surface', Locatable)  # Superficie (Tipo de ubicable)
Pallet = UserType('Pallet', Surface)  # Palé (Tipo de superficie)
Crate = UserType('Crate', Surface)  # Caja (Tipo de superficie)

for tipo_de_objeto in [Place, Locatable, Depot, Distributor, Truck, Hoist, Surface, Pallet, Crate]:
    dominio_depot.user_types.append(tipo_de_objeto)

In [15]:
# Predicados booleanos

# El predicado AT representa que el ubicable x está en el lugar y
at = Fluent('AT', BoolType(), x=Locatable, y=Place)
# El predicado ON representa que la caja x está sobre la superficie y
on = Fluent('ON', BoolType(), x=Crate, y=Surface)
# El predicado IN representa que la caja x está en el camión y
# (Nótese el guión bajo incluido en el nombre de la variable, ya que no se
# puede usar in, al tratarse de un identificador reservado de Python)
in_ = Fluent('IN', BoolType(), x=Crate, y=Truck)
# El predicado LIFTING representa que el polipasto x está levantando la caja y
lifting = Fluent('LIFTING', BoolType(), x=Hoist, y=Crate)
# El predicado AVAILABLE representa que el polipasto x está disponible
available = Fluent('AVAILABLE', BoolType(), x=Hoist)
# El predicado CLEAR representa que la superficie x está despejada
clear = Fluent('CLEAR', BoolType(), x=Surface)

for fluente in [at, on, in_, lifting, available, clear]:
    dominio_depot.add_fluent(fluente, default_initial_value=False)

In [ ]:
# Esquemas de acciones

# La acción DRIVE representa que el camión x va del lugar y al lugar z
drive = InstantaneousAction('DRIVE', x=Truck, y=Place, z=Place)
x = drive.x
y = drive.y
z = drive.z
for hecho in [at(x,y)]:
    drive.add_precondition(hecho)
for hecho in [at(x,y)]:
    drive.add_effect(hecho, False)
for hecho in [at(x,z)]:
    drive.add_effect(hecho, True)

drive
# La acción LIFT representa que el polipasto x levanta la caja y que se
# encontraba sobre la superficie z en el lugar p
lift = InstantaneousAction('LIFT', x=Hoist, y=Crate, z=Surface, p=Place)
x = lift.x
y = lift.y
z = lift.z
p = lift.p
for hecho in [at(x,p), at(z,p), on(y,z), available(x), clear(y)]:
    lift.add_precondition(hecho)
for hecho in [available(x), on(y,z), clear(y)]:
    lift.add_effect(hecho, False)
for hecho in [lifting(x,y),clear(z)]:
    lift.add_effect(hecho, True)

lift

# La acción DROP representa que el polipasto x deja la caja y sobre la
# superficie z en el lugar p
drop = InstantaneousAction('DROP', x=Hoist, y=Crate, z=Surface, p=Place)
x = drop.x
y = drop.y
z = drop.z
p = drop.p
for hecho in [at(x,p), at(z,p), lifting(x,y),clear(z)]:
    drop.add_precondition(hecho)
for hecho in [clear(z), lifting(x,y)]:
    drop.add_effect(hecho, False)
for hecho in [clear(y), available(x), on(y,z), at(y,p)]:
    drop.add_effect(hecho, True)

# La acción LOAD representa que el polipasto x carga la caja y en el
# camión z en el lugar p
load = InstantaneousAction('LOAD', x=Hoist, y=Crate, z=Truck, p=Place)
x = load.x
y = load.y
z = load.z
p = load.p
for hecho in [at(x,p), at(y,p), at(z,p),lifting(x,y)]:
    load.add_precondition(hecho)
for hecho in [lifting(x,y)]:
    load.add_effect(hecho, False)
for hecho in [available(x), in_(y,z)]:
    load.add_effect(hecho, True)

load

# La acción UNLOAD representa que el polipasto x descarga la caja y del
# camión z en el lugar p
unload = InstantaneousAction('UNLOAD', x=Hoist, y=Crate, z=Truck, p=Place)
x = unload.x
y = unload.y
z = unload.z
p = unload.p
for hecho in [at(x,p), at(z,p), in_(y,z), available(x)]:
    unload.add_precondition(hecho)
for hecho in [in_(y,z),available(x)]:
    unload.add_effect(hecho, False)
for hecho in [lifting(x,y)]:
    unload.add_effect(hecho, True)



dominio_depot.add_actions([drive, lift, drop, load, unload])

In [41]:
from unified_planning.io import PDDLReader
from unified_planning.shortcuts import OneshotPlanner
from unified_planning.engines import PlanGenerationResultStatus
import time

lector_PDDL = PDDLReader()
up.shortcuts.get_environment().credits_stream = None

lista_archivos = [ f"Depot/pfile{i}" for i in range(1,23) ]
for archivo in lista_archivos:

    problema = lector_PDDL.parse_problem('Depot/dominio_problem.pddl',archivo)
    
    with OneshotPlanner(name='fast-downward') as planner:
        inicio = time.time()
        resultado = planner.solve(problema,timeout=30)
        fin = time.time()
        tiempo_total = fin - inicio
        print(f"Resultado para {archivo}: {resultado.status}")
        print(f"tiempo empleado: {tiempo_total:.2f}")
        if resultado.status == PlanGenerationResultStatus.SOLVED_SATISFICING:
            print(f"Longitud del plan {len(resultado.plan.actions)}")
        else:
            print(PlanGenerationResultStatus.TIMEOUT)
        
          


Resultado para Depot/pfile1: PlanGenerationResultStatus.SOLVED_SATISFICING
tiempo empleado: 0.47
Longitud del plan 4
Resultado para Depot/pfile2: PlanGenerationResultStatus.SOLVED_SATISFICING
tiempo empleado: 0.57
Longitud del plan 6
Resultado para Depot/pfile3: PlanGenerationResultStatus.SOLVED_SATISFICING
tiempo empleado: 0.58
Longitud del plan 12
Resultado para Depot/pfile4: PlanGenerationResultStatus.SOLVED_SATISFICING
tiempo empleado: 0.87
Longitud del plan 12
Resultado para Depot/pfile5: PlanGenerationResultStatus.SOLVED_SATISFICING
tiempo empleado: 0.78
Longitud del plan 18
Resultado para Depot/pfile6: PlanGenerationResultStatus.SOLVED_SATISFICING
tiempo empleado: 1.27
Longitud del plan 22
Resultado para Depot/pfile7: PlanGenerationResultStatus.SOLVED_SATISFICING
tiempo empleado: 0.59
Longitud del plan 10
Resultado para Depot/pfile8: PlanGenerationResultStatus.SOLVED_SATISFICING
tiempo empleado: 0.80
Longitud del plan 14
Resultado para Depot/pfile9: PlanGenerationResultStatus.SO

### Ejercicio 3

[Sokoban](https://en.wikipedia.org/wiki/Sokoban) es un videojuego clásico de tipo puzle. En este juego el objetivo es empujar cajas, u otro tipo de objetos, en un almacén hasta llevarlos a las ubicaciones de almacenamiento. El juego se ve desde una perspectiva cenital. Los objetos solo se pueden empujar, nunca tirar de ellos, y solo un objeto se puede empujar a la vez. El desafío principal es planificar movimientos correctamente para evitar causar un punto muerto, una situación en la que un objeto o el jugador queda atrapado permanentemente, haciendo que el rompecabezas sea irresoluble.

En este ejercicio se pide lo siguiente:

1. Construir un dominio de planificación automática para el juego del Sokoban. Ese dominio debe contener los siguientes elementos:
   * Tipos de objetos: `thing`, `location`, `direction`, `player` (subtipo de `thing`), `stone` (subtipo de `thing`).
   * Predicados:
     * `CLEAR`: representa que una determinada localización (`location`) no contiene ninguna cosa (`thing`).
     * `AT`: representa que una cosa (`thing`) está en una determinada localización (`location`).
     * `AT-GOAL`: representa que una piedra (`stone`) está en una localización objetivo.
     * `IS-GOAL`: representa que una determinada localización (`location`) es una localización objetivo.
     * `IS-NONGOAL`: representa que una determinada localización (`location`) no es una localización objetivo.
     * `MOVE-DIR`: representa que se puede pasar de una determinada localización (`location`) a otra localización (`location`) adyacente moviéndose en una cierta dirección (`direction`).
   * Acciones:
     * `MOVE`: representa que el jugador (`player`) se mueve de la localización (`location`) que ocupa a una localización (`location`) libre adyacente en una determinada dirección (`direction`).
     * `PUSH-TO-NONGOAL`: representa que el jugador (`player`), estando en una determinada localización (`location`), empuja una piedra (`stone`) desde una localización (`location`) a otra localización (`location`) libre adyacente, que no es una localización objetivo, en una determinada dirección (`direction`).
     * `PUSH-TO-GOAL`: representa que el jugador (`player`), estando en una determinada localización (`location`), empuja una piedra (`stone`) desde una localización (`location`) a otra localización (`location`) libre adyacente, que es una localización objetivo, en una determinada dirección (`direction`).

In [ ]:
from unified_planning.shortcuts import *
from unified_planning.io import PDDLWriter, PDDLReader
from unified_planning.engines import PlanGenerationResultStatus

# =====================================================================
# PARTE 1: CONSTRUIR EL DOMINIO (Cumpliendo el apartado 1 del Ejercicio)
# =====================================================================
# Nombramos el dominio en minúsculas para que coincida con las instancias
dominio_sokoban = Problem("sokoban")

# 1. Tipos de objetos
Thing = UserType("thing")
Location = UserType("location")
Direction = UserType("direction")
Player = UserType("player", father=Thing)
Stone = UserType("stone", father=Thing)

# 2. Fluentes (Predicados limpios, sin números)
clear = Fluent("clear", BoolType(), l=Location)
at = Fluent("at", BoolType(), t=Thing, l=Location)
at_goal = Fluent("at-goal", BoolType(), s=Stone)
is_goal = Fluent("is-goal", BoolType(), l=Location)
is_nongoal = Fluent("is-nongoal", BoolType(), l=Location)
move_dir = Fluent("move-dir", BoolType(), l1=Location, l2=Location, d=Direction)

dominio_sokoban.add_fluents([clear, at, at_goal, is_goal, is_nongoal, move_dir])

# 3. Acción: MOVE
move = InstantaneousAction("move", p=Player, l1=Location, l2=Location, d=Direction)
p, l1, l2, d = move.parameters
move.add_precondition(at(p, l1))
move.add_precondition(move_dir(l1, l2, d))
move.add_effect(at(p, l1), False)
move.add_effect(at(p, l2), True)

# 4. Acción: PUSH-TO-NONGOAL
push_nogoal = InstantaneousAction("push-to-nogoal", p=Player, l1=Location, s=Stone, l2=Location, d=Direction)
p, l1, s, l2, d = push_nogoal.parameters
push_nogoal.add_precondition(at(p, l1))
push_nogoal.add_precondition(at(s, l1))
push_nogoal.add_precondition(move_dir(l1, l2, d))
push_nogoal.add_precondition(clear(l2))
push_nogoal.add_precondition(is_nongoal(l2))
push_nogoal.add_precondition(is_nongoal(l1))
# Efectos
push_nogoal.add_effect(at(p, l1), False)
push_nogoal.add_effect(at(s, l1), False)
push_nogoal.add_effect(clear(l2), False)
push_nogoal.add_effect(at_goal(s), False)
push_nogoal.add_effect(at(p, l2), True)
push_nogoal.add_effect(at(s, l2), True)

# 5. Acción: PUSH-TO-GOAL
push_goal = InstantaneousAction("push-to-goal", p=Player, l1=Location, s=Stone, l2=Location, d=Direction)
p, l1, s, l2, d = push_goal.parameters
push_goal.add_precondition(at(p, l1))
push_goal.add_precondition(at(s, l1))
push_goal.add_precondition(clear(l2))
push_goal.add_precondition(move_dir(l1, l2, d))
push_goal.add_precondition(is_nongoal(l1))
push_goal.add_precondition(is_goal(l2))
# Efectos
push_goal.add_effect(at(p, l1), False)
push_goal.add_effect(at(s, l1), False)
push_goal.add_effect(clear(l2), False)
push_goal.add_effect(at(p, l2), True)
push_goal.add_effect(at(s, l2), True)
push_goal.add_effect(at_goal(s), True)

dominio_sokoban.add_actions([move, push_nogoal, push_goal])

# Exportamos nuestro dominio limpio a un PDDL temporal
PDDLWriter(dominio_sokoban).write_domain("Sokoban/dominio_limpio.pddl")



2. Usar el algoritmo $\mathrm{A}^{*}$ y la heurística $h^{\mathrm{max}}$ para resolver, con la menor cantidad posible de movimientos de empuje, los puzles que se encuentran en la carpeta Sokoban. Para ello, asignar coste $1$ a las acciones `PUSH-TO-NONGOAL` y `PUSH-TO-GOAL` y coste $0$ al resto de acciones.

In [74]:
# =====================================================================
# PARTE 2: LEER, CONFIGURAR Y RESOLVER
# =====================================================================

# Leemos la instancia UNIÉNDOLA a nuestro dominio limpio (usamos barras /)
lector = PDDLReader()
problema_instancia = lector.parse_problem("Sokoban/dominio_limpio.pddl", "Sokoban/p01.pddl")

# Configuramos la métrica de costes como pide la práctica
coste_de_empujar = Fluent('COSTE-DE-EMPUJAR', IntType(),
                          p=Player, l1=Location, s=Stone, f=Location, l2=Location, d=Direction)
problema_instancia.add_fluent(coste_de_empujar, default_initial_value=Int(1))

métrica = MinimizeActionCosts(
    {push_nogoal: coste_de_empujar(
        push_nogoal.p, push_nogoal.l1, push_nogoal.s,
        push_nogoal.f, push_nogoal.l2, push_nogoal.d),
     push_goal: coste_de_empujar(
        push_goal.p, push_goal.l1, push_goal.s,
        push_goal.f, push_goal.l2, push_goal.d)},
    default=Int(0))

problema_instancia.add_quality_metric(métrica)

# Usamos A* y hmax nativos de Fast Downward
parametros_motor = {"fast_downward_search_config": "astar(hmax())"}

with OneshotPlanner(name="fast-downward", params=parametros_motor) as planner:
    print("Pensando... (A* evaluando heurística)")
    resultado = planner.solve(problema_instancia, timeout=60)

    if resultado.status in [PlanGenerationResultStatus.SOLVED_SATISFICING, 
                            PlanGenerationResultStatus.SOLVED_OPTIMALLY]:
        
        coste_total = sum(costes_acciones[paso.action] for paso in resultado.plan.actions)
        
        print(f"\n✅ ¡Puzle Resuelto!")
        print(f"🔹 Movimientos de empuje (Coste): {coste_total}")
        print(f"🔹 Pasos totales del jugador: {len(resultado.plan.actions)}")
        print("\n--- SECUENCIA ---")
        for i, paso in enumerate(resultado.plan.actions):
            print(f"{i+1}. {paso}")
    else:
        print(f"❌ No resuelto (Estado: {resultado.status.name})")




AttributeError: Transition 'push-to-nogoal' has no attribute or parameter 'f'